# DuckDB SQL syntax sugar: familiar SQL, less scaffolding
Write common analytical queries with less repetition.
<br>Explore `GROUP BY ALL`, `EXCLUDE`, `REPLACE`, `RENAME`, `QUALIFY`, aliases, and `PIVOT`.
<br>Additional examples follow sections 1–5.
**Before you begin:**
1. Use the Pure Python runtime and attach `lh_pond` as the default Lakehouse.
2. Run `0. Setup` with `DATA_SCOPE = "one_year"`.
3. Run this notebook’s setup cell; it creates its own DuckDB connection.


## Setup
Use the same flight files as Notebook 1.
<br>`FLIGHT_FILES` holds only the file path. Each query shows `read_parquet()` directly.
<br>The `f` before the SQL string inserts that path; `.show()` displays the result.
<br>`union_by_name = true` aligns columns by name across files.


In [ ]:
import duckdb

FLIGHT_FILES = "/lakehouse/default/Files/flights_eu/one_year/flight_list_*.parquet"

con = duckdb.connect()

print("Flight files:", FLIGHT_FILES)


### Check the demo files
Confirm that the flight folder contains matching Parquet files.
<br>If this check fails, run `0. Setup` with `DATA_SCOPE = "one_year"`.


In [ ]:
from glob import glob

assert glob(FLIGHT_FILES), (
    "Run 0. Setup with DATA_SCOPE = 'one_year' before opening this notebook."
)
print("Demo files found.")


## 1. `GROUP BY ALL`: Group by the columns you select
Count flights for each origin (`adep`) and destination (`ades`) pair.
<br>`GROUP BY ALL` groups by the selected expressions that are not aggregates.
<br>Here, it replaces `GROUP BY adep, ades`.
<br>The result shows the ten busiest origin–destination pairs.


In [ ]:
con.sql(f"""
    SELECT adep,
           ades,
           count(*) AS flights
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    WHERE adep IS NOT NULL
      AND ades IS NOT NULL
    GROUP BY ALL
    ORDER BY flights DESC
    LIMIT 10
""").show()


## 2. Shape a result without rewriting every column
The `WITH` block selects five columns to keep the display narrow.
<br>The outer query demonstrates three shortcuts:
- `EXCLUDE` removes the flight ID.
- `REPLACE` converts `dof` from text to a date while keeping its name. Unparseable values become NULL.
- `RENAME` gives the airport columns clearer names.
The final result shows the date, origin, destination, and aircraft type.
<br>These changes affect the query result; the source Parquet files are unchanged.


In [ ]:
con.sql(f"""
    WITH flights AS (
        SELECT flt_id, dof, adep, ades, typecode
        FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
        WHERE adep IS NOT NULL
          AND ades IS NOT NULL
          AND typecode IS NOT NULL
    )
    SELECT * EXCLUDE (flt_id)
    REPLACE (try_cast(dof AS DATE) AS dof)
    RENAME (adep AS origin_airport, ades AS destination_airport)
    FROM flights
    LIMIT 10
""").show()


## 3. `QUALIFY`: Keep the top route from each origin
**Keep the busiest route for each origin.**
<br>`row_number()` ranks routes; `QUALIFY` keeps rank 1 without another query layer.
<br>Destination code breaks ties. `LIMIT 10` keeps the display short.


In [ ]:
con.sql(f"""
    SELECT adep,
           ades,
           count(*) AS flights,
           row_number() OVER (
               PARTITION BY adep
               ORDER BY count(*) DESC, ades
           ) AS route_rank
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    WHERE adep IS NOT NULL
      AND ades IS NOT NULL
    GROUP BY adep, ades
    QUALIFY route_rank = 1
    ORDER BY flights DESC
    LIMIT 10
""").show()


## 4. Alias reuse: name a calculation once
**Name the count `flights`, then reuse it.**
<br>DuckDB lets `HAVING` use the `flights` alias instead of repeating `count(*)`.
<br>Keep aircraft types with more than 100,000 flights, then sort by that count.


In [ ]:
con.sql(f"""
    SELECT typecode,
           count(*) AS flights
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    WHERE typecode IS NOT NULL
    GROUP BY ALL
    HAVING flights > 100000
    ORDER BY flights DESC
""").show()


## 5. Dynamic `PIVOT`: turn months into columns
**Turn month values into columns, with one row per airport.**
<br>Use January–March 2025 to keep the result readable on screen.
<br>`ON month` chooses the column values; `USING sum(flights)` fills the cells.
<br>DuckDB discovers the month columns from the data. Missing combinations appear as NULL.


In [ ]:
con.sql(f"""
    PIVOT (
        SELECT adep,
               strftime(try_cast(dof AS DATE), '%Y-%m') AS month,
               count(*) AS flights
        FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
        WHERE adep IN ('LEBL', 'EDDF', 'EGLL', 'EHAM', 'LFPG')
          AND try_cast(dof AS DATE) >= DATE '2025-01-01'
          AND try_cast(dof AS DATE) < DATE '2025-04-01'
        GROUP BY ALL
    )
    ON month
    USING sum(flights)
    GROUP BY adep
    ORDER BY adep
""").show()


## 6. `UNION ALL BY NAME`: match columns by name
**Additional example**
**Stack results by column name, even when their order differs.**
<br>Notice that `role` comes first in one query and last in the other.
<br>`ALL` keeps duplicates; the final result shows ten airport-and-role combinations.


In [ ]:
con.sql(f"""
    SELECT 'origin' AS role,
           adep AS airport,
           count(*) AS flights
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    WHERE adep IS NOT NULL
    GROUP BY ALL

    UNION ALL BY NAME

    SELECT ades AS airport,
           count(*) AS flights,
           'destination' AS role
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    WHERE ades IS NOT NULL
    GROUP BY ALL

    ORDER BY flights DESC
    LIMIT 10
""").show()


## 7. `COLUMNS(...)`: find related fields
The flight files have several time-related columns. `COLUMNS('seen|time')` selects
columns whose names contain `seen` or `time`, such as `first_seen` and `unix_time`.
<br>The `|` means “or” in this column-name pattern.
This is useful when inspecting a wide file and you do not yet know every field name.


In [ ]:
con.sql(f"""
    SELECT COLUMNS('seen|time')
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    LIMIT 10
""").show()


## Bonus: start with `FROM` and return a small fraction
**Start with the source, then choose the columns.**
<br>`LIMIT 0.001%` limits the result size; it does not select a random sample.
<br>Row order is unspecified. A small result can still require a large scan.


In [ ]:
con.sql(f"""
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    SELECT adep,
           ades,
           typecode
    WHERE adep IS NOT NULL
      AND ades IS NOT NULL
      AND typecode IS NOT NULL
    LIMIT 0.001%
""").show()


## Bonus: profile columns with `SUMMARIZE`
**Inspect column types, ranges, and missing values.**
<br>`DESCRIBE` reports the schema. `SUMMARIZE` also examines the data.
<br>Keep five profile fields for readability. Text-date ranges do not validate date parsing.


In [ ]:
con.sql(f"""
    SELECT column_name,
           column_type,
           min,
           max,
           null_percentage
    FROM (
        SUMMARIZE
        SELECT dof, adep, ades, typecode, first_seen, last_seen
        FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    )
""").show()


## Takeaway
DuckDB lets you query Lakehouse files with familiar SQL. These conveniences shorten
common analytical queries while keeping the logic visible.
<br>When moving the SQL to another engine, check support for these conveniences
and adapt the syntax while preserving the query’s meaning.
